In [1]:
from llama_index.llms.groq import Groq
from llama_index.core import Settings

Settings.llm = Groq(model="openai/gpt-oss-20b",
                    context_window=8000)
response = Settings.llm.complete("Explain the importance of low latency LLMs")
print(response)

/home/arthas/anaconda3/envs/tcc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from llama_index.core.agent.workflow import AgentWorkflow

def sum(a, b):
    """useful for sum numbers"""
    return a + b
    
def multiply(a, b):
    """useful for multiply numbers"""
    return a * b
    
agent = AgentWorkflow.from_tools_or_functions(
        [sum, multiply],
        llm=Settings.llm,
        system_prompt="You are a helpful assistant that can do calculations"
)

response = await agent.run("What is 7 + 5 * 3?")
print(response)

# But the model stills responds to other contexts
response = await agent.run("Which is the name of Germany capital?")
print(response)

The result is **22**.
The capital of Germany is **Berlin**.


In [3]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

path = "./models/nomic-ai-v1.5"
# downloading the model
#
# from sentence_transformers import SentenceTransformer
#
# model = SentenceTransformer(
#     "nomic-ai/nomic-embed-text-v1.5",
#     trust_remote_code=True
# )
# 
# model.save(path)

Settings.embed_model = HuggingFaceEmbedding(
    model_name=path,
    trust_remote_code=True
)

documents = SimpleDirectoryReader("data_test").load_data()

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(
    response_mode="refine"
)

async def search_document(query: str) -> str:
    """useful for search about country's presidents"""
    response = await query_engine.aquery(query)
    return str(response)

# It'll restrict the model on another level
system_prompt = """ 
    ever, EVER, use only the search_document to give responses.
    Responds naturally, without marks that you viewed a document.
"""

agent = AgentWorkflow.from_tools_or_functions(
    [search_document],
    llm=Settings.llm,
    system_prompt=system_prompt
)

async def main():
    # In the 'documents = SimpleDirectoryReader("data_test").load_data()',
    # the information inside is: The Mongolia president at 1988
    # was Arthas Brunnett Mangueira
    response = await agent.run("What was the name of the Mongolia president at 1988?",
                               max_iterations=10, early_stopping_method="generate")
    print(response)

await main()

I0000 00:00:1778437230.864404   72144 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778437230.895042   72144 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
<All keys matched successfully>
2026-05-10 15:20:34,678 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 15:20:35,069 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 15:20:35,257 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The president of Mongolia in 1988 was Arthas Brunnett Mangueira.


In [4]:
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
import chromadb

db = chromadb.PersistentClient(path="./chromadb-teste")
collection = db.get_or_create_collection("star-wars")
vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

if collection.count() == 0:
    documents = SimpleDirectoryReader("star-wars").load_data()
    index = VectorStoreIndex.from_documents(
        documents, storage_context=storage_context
    )
else:
    index = VectorStoreIndex.from_vector_store(
        vector_store=vector_store, storage_context=storage_context
    )

query_engine = index.as_query_engine()

async def search_star_wars(query: str) -> str:
    """useful to search for star wars contents"""
    response = await query_engine.aquery(query)
    return str(response)

agent = AgentWorkflow.from_tools_or_functions(
    [search_star_wars],
    llm=Settings.llm,
    system_prompt="""Only responds questions about the given document.
                     Always say when you found the information using search_star_wars or not"""
)

async def main():
    response = await agent.run("Who is the main's character?")
    print(str(response))
    response = await agent.run("Who is the Deak's and Annikin Starkiller's father?")
    print(str(response))

await main()

E0000 00:00:1778437235.828592   72144 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1778437235.828825   72144 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1778437235.828826   72144 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1778437235.828827   72144 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1778437235.828829   72144 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.
2026-05-10 15:20:36,232 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


I’m sorry, but I don’t have a document to reference for that question. I couldn’t find any relevant information using `search_star_wars`.


2026-05-10 15:20:36,625 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 15:20:37,451 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 15:20:37,639 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 15:20:38,271 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-10 15:20:38,501 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


I found the information using **search_star_wars**.  
Both Deak Starkiller and Annikin Starkiller are sons of **Kane Starkiller**.


In [12]:
from llama_index.core.node_parser import SentenceSplitter

documents = SimpleDirectoryReader("star-wars").load_data()

# A example of how the parse works
parser = SentenceSplitter(chunk_size=128, chunk_overlap=30)
nodes = parser.get_nodes_from_documents(documents)

for i, node in enumerate(nodes[:3]): 
    print(f"--- Node {i} (Size: {len(node.get_content())} characters) ---")
    print(node.get_content())
    print(f"Metadata: {node.metadata}\n")

--- Node 0 (Size: 337 characters) ---
The Jedi Kane Starkiller and his 2 sons, Deak and Annikin Starkiller, live on the 4th Moon of Utapau. Their seclusion from the forces of the Galactic Empire is interrupted by the arrival of a Sith flying a Banta Four starfighter. The Jedi and his sons go to investigate, but the Sith gets a jump on them, killing Deak with a single blow.
Metadata: {'file_path': '/home/arthas/Documents/TCC/Cook-master/star-wars/Star-wars.txt', 'file_name': 'Star-wars.txt', 'file_type': 'text/plain', 'file_size': 869, 'creation_date': '2026-05-10', 'last_modified_date': '2026-05-10'}

--- Node 1 (Size: 372 characters) ---
The Jedi and his sons go to investigate, but the Sith gets a jump on them, killing Deak with a single blow. However, their foe is no match for a Jedi Master, and Kane avenges his apprentice. The exiles must flee the Kessil system and leave for their homeworld of Aquilae. Aquilae, a planet not yet controlled by the Empire, is ruled by the wise King Kay